# Capítulo 8. Random Forest

**Aprendizaje y Clasificación Automática con R**  
**Autor:** Jesús Gilberto Rodríguez Escobedo

Este cuaderno es **independiente y autónomo**: puede abrirse directamente sin ejecutar capítulos anteriores.

1. Ejecute primero la celda **Preparación automática y autónoma del capítulo**.
2. Después ejecute las celdas en orden.
3. Si Colab reinicia la sesión, vuelva a ejecutar desde la primera celda.

[Volver al índice de cuadernos Colab](https://colab.research.google.com/github/gilbertorodriguez59/libro-machine-learning-r-covid/blob/main/colab/00-indice-colabs.ipynb)


In [ ]:
# Preparación automática y autónoma del capítulo
options(repos = c(CRAN = "https://cloud.r-project.org"))

paquetes_libro <- c(
  "ggplot2", "readr", "dplyr", "tidyr", "stringr", "data.table",
  "class", "rpart", "randomForest", "ranger", "e1071", "naivebayes",
  "neuralnet", "cluster", "caret", "factoextra", "scales", "plotly", "DT"
)
faltantes <- paquetes_libro[!vapply(paquetes_libro, requireNamespace, logical(1), quietly = TRUE)]
if (length(faltantes)) install.packages(faltantes)

dir.create("datos/covid19/procesados", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/muestras", showWarnings = FALSE, recursive = TRUE)
dir.create("datos/covid19/diccionarios", showWarnings = FALSE, recursive = TRUE)

archivos_colab <- c(
  "util_graficas.R" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/util_graficas.R",
  "datos/atus_ml_preparado.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/atus_ml_preparado.csv",
  "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz",
  "datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/muestras/covid19_mexico_2022_muestra.csv.gz",
  "datos/covid19/diccionarios/diccionario_covid19_ml.csv" = "https://raw.githubusercontent.com/gilbertorodriguez59/libro-machine-learning-r-covid/main/datos/covid19/diccionarios/diccionario_covid19_ml.csv"
)
for (destino in names(archivos_colab)) {
  if (!file.exists(destino)) download.file(archivos_colab[[destino]], destino, mode = "wb", quiet = TRUE)
}
stopifnot(all(file.exists(names(archivos_colab))))
source("util_graficas.R")
cat("Entorno autónomo listo. R:", R.version.string, "\n")


# Random Forest para clasificación

La formulación matemática de **bootstrap, ensambles y Random Forest** se desarrolla con mayor profundidad
en los capítulos 7 y 14 de *Fundamentos Matemáticos del Aprendizaje
Automático* [@rodriguez2026fundamentos].

## Objetivos del capítulo

Al finalizar este capítulo, el lector será capaz de explicar Random Forest, entrenar un modelo en R, interpretar importancia de variables y evaluar desempeño.

## Introducción

Random Forest combina muchos árboles de decisión. Cada árbol aprende sobre una muestra aleatoria de los datos y la predicción final se obtiene por votación mayoritaria.

## Fundamento matemático

Si se construyen $B$ árboles, la predicción final es:

$$
\hat{y}(x) = moda\{\hat{y}^{(1)}(x), \hat{y}^{(2)}(x), \ldots, \hat{y}^{(B)}(x)\}
$$

## Cargar paquetes y datos


In [ ]:
library(readr)
library(dplyr)
library(ggplot2)
library(ranger)
source("util_graficas.R")

ruta_atus_ml <- "datos/atus_ml_preparado.csv"

if (file.exists(ruta_atus_ml)) {
  atus_ml <- read_csv(ruta_atus_ml, show_col_types = FALSE)
  print("Base preparada cargada correctamente.")
} else {
  atus_ml <- NULL
  print("No se encontró el archivo datos/atus_ml_preparado.csv. Ejecuta primero el capítulo 2.")
}


## Muestra de trabajo


In [ ]:
if (!is.null(atus_ml)) {
  set.seed(123)
  atus_rf_base <- atus_ml |>
    sample_n(min(30000, nrow(atus_ml))) |>
    mutate(
      accidente_con_victimas = factor(accidente_con_victimas, levels = c("Con víctimas", "Solo daños")),
      MES = as.factor(MES),
      ID_HORA = as.numeric(ID_HORA),
      DIASEMANA = as.factor(DIASEMANA),
      TIPACCID = as.factor(TIPACCID),
      CAUSAACCI = as.factor(CAUSAACCI)
    ) |>
    na.omit()

  data.frame(filas = nrow(atus_rf_base), columnas = ncol(atus_rf_base))
}

if (exists("atus_rf_base")) {
  ggplot(atus_rf_base, aes(x = accidente_con_victimas, fill = accidente_con_victimas)) +
    geom_bar(width = 0.7) +
    escala_clases_fill() +
    scale_y_continuous(labels = etiqueta_numero) +
    labs(
      title = "Distribución de la variable respuesta",
      subtitle = "Muestra para Random Forest",
      x = "Clase",
      y = "Número de accidentes"
    ) +
    tema_libro() +
    theme(legend.position = "none")
}


## Explicación del código
Se toma una muestra de trabajo para mantener el tiempo de ejecución razonable. El modelo Random Forest suele ser más pesado que un solo árbol.

## División en entrenamiento y prueba


In [ ]:
if (exists("atus_rf_base")) {
  set.seed(123)
  idx <- sample(1:nrow(atus_rf_base), size = round(0.7 * nrow(atus_rf_base)))
  entrenamiento <- atus_rf_base[idx, ]
  prueba <- atus_rf_base[-idx, ]

  data.frame(conjunto = c("Entrenamiento", "Prueba"), registros = c(nrow(entrenamiento), nrow(prueba)))
}


## Ajustar Random Forest


In [ ]:
if (exists("entrenamiento")) {
  set.seed(123)

  modelo_rf <- ranger(
    accidente_con_victimas ~ MES + ID_HORA + DIASEMANA + TIPACCID + CAUSAACCI,
    data = entrenamiento,
    num.trees = 200,
    mtry = 3,
    min.node.size = 20,
    importance = "impurity",
    probability = TRUE,
    seed = 123
  )

  modelo_rf
}


## Interpretación del resultado
El resumen del modelo muestra cuántos árboles se construyeron, cuántas variables se probaron en cada división y el error fuera de bolsa.

## Importancia de variables


In [ ]:
if (exists("modelo_rf")) {
  importancia_rf <- data.frame(
    variable = names(modelo_rf$variable.importance),
    importancia = as.numeric(modelo_rf$variable.importance)
  ) |>
    arrange(desc(importancia))

  importancia_rf
}

if (exists("importancia_rf")) {
  ggplot(importancia_rf, aes(x = reorder(variable, importancia), y = importancia)) +
    geom_col(fill = col_azul, width = 0.75) +
    coord_flip() +
    labs(title = "Importancia de variables en Random Forest", x = "Variable", y = "Importancia") +
    tema_libro()
}


## Explicación del código
La importancia de variables muestra qué predictores aportan más a la reducción de impureza en los árboles del bosque.

## Predicción y métricas


In [ ]:
if (exists("modelo_rf")) {
  predicciones_rf <- predict(modelo_rf, data = prueba)
  probabilidades_rf <- predicciones_rf$predictions[, "Con víctimas"]
  clases_rf <- colnames(predicciones_rf$predictions)[max.col(predicciones_rf$predictions, ties.method = "first")]
  clases_rf <- factor(clases_rf, levels = c("Con víctimas", "Solo daños"))
  real_rf <- factor(prueba$accidente_con_victimas, levels = c("Con víctimas", "Solo daños"))

  matriz_confusion_rf <- table(Real = real_rf, Predicho = clases_rf)
  matriz_confusion_rf
}

if (exists("matriz_confusion_rf")) {
  VP <- matriz_confusion_rf["Con víctimas", "Con víctimas"]
  FN <- matriz_confusion_rf["Con víctimas", "Solo daños"]
  FP <- matriz_confusion_rf["Solo daños", "Con víctimas"]
  VN <- matriz_confusion_rf["Solo daños", "Solo daños"]

  data.frame(exactitud=(VP+VN)/(VP+FN+FP+VN), sensibilidad=VP/(VP+FN), especificidad=VN/(VN+FP))
}


## Interpretación del resultado
La sensibilidad indica qué tan bien detecta accidentes con víctimas. La especificidad indica qué tan bien reconoce accidentes de solo daños.

## Distribución de probabilidades predichas


In [ ]:
if (exists("probabilidades_rf")) {
  datos_prob_rf <- data.frame(probabilidad = probabilidades_rf, clase_real = real_rf)

  ggplot(datos_prob_rf, aes(x = probabilidad, fill = clase_real)) +
    geom_histogram(bins = 30, alpha = 0.75, position = "identity") +
    escala_clases_fill(name = "Clase real") +
    labs(
      title = "Distribución de probabilidades predichas",
      subtitle = "Random Forest",
      x = "Probabilidad predicha de 'Con víctimas'",
      y = "Frecuencia"
    ) +
    tema_libro()
}


## Laboratorio interactivo: votación de un bosque

Este simulador muestra cómo múltiples árboles emiten votos y cómo la predicción
del bosque se estabiliza al aumentar el número de árboles.


**Laboratorio interactivo:** este bloque se ejecuta en la versión web mediante Shinylive; aquí se conserva el desarrollo reproducible del capítulo.


### Laboratorio disponible en la versión web

El laboratorio permite cambiar el número de árboles y observar cómo se
estabiliza la proporción acumulada de votos del bosque.

## Caso aplicado B: Random Forest con COVID-19

Ahora aplicamos un bosque aleatorio a la misma muestra COVID utilizada en k-NN y árboles. Esto permite comparar un árbol individual con un ensamble de muchos árboles.

> **Uso académico:** las importancias y predicciones corresponden a una muestra educativa y no deben usarse para decisiones médicas.


In [ ]:
ruta_covid <- "datos/covid19/procesados/covid19_mexico_2022_ml_preparado.csv.gz"

if (file.exists(ruta_covid)) {
  covid_rf <- read_csv(ruta_covid, show_col_types = FALSE) |>
    select(MURIO, EDAD, NEUMONIA, DIABETES, HIPERTENSION,
           OBESIDAD, RENAL_CRONICA, NUM_COMORBILIDADES) |>
    tidyr::drop_na()

  set.seed(2026)
  covid_rf <- covid_rf |>
    sample_n(min(12000, nrow(covid_rf))) |>
    mutate(MURIO = factor(MURIO, levels = c(1, 0), labels = c("Defunción", "Sin defunción")))

  set.seed(2026)
  idx_covid <- sample(seq_len(nrow(covid_rf)), size = floor(0.80 * nrow(covid_rf)))
  covid_train <- covid_rf[idx_covid, ]
  covid_test <- covid_rf[-idx_covid, ]
}


In [ ]:
if (exists("covid_train")) {
  rf_covid <- ranger(
    MURIO ~ EDAD + NEUMONIA + DIABETES + HIPERTENSION + OBESIDAD + RENAL_CRONICA + NUM_COMORBILIDADES,
    data = covid_train,
    num.trees = 300,
    mtry = 3,
    min.node.size = 20,
    importance = "impurity",
    probability = TRUE,
    seed = 2026
  )
  rf_covid
}


In [ ]:
if (exists("rf_covid")) {
  imp_covid_rf <- data.frame(
    variable = names(rf_covid$variable.importance),
    importancia = as.numeric(rf_covid$variable.importance)
  ) |>
    arrange(desc(importancia))

  imp_covid_rf

  ggplot(imp_covid_rf, aes(x = reorder(variable, importancia), y = importancia)) +
    geom_col(fill = col_azul, width = 0.75) +
    coord_flip() +
    labs(title = "Importancia de variables: Random Forest COVID-19", x = "Variable", y = "Importancia") +
    tema_libro()
}


In [ ]:
if (exists("rf_covid")) {
  pred_prob_covid_rf <- predict(rf_covid, data = covid_test)$predictions
  pred_clase_covid_rf <- colnames(pred_prob_covid_rf)[max.col(pred_prob_covid_rf, ties.method = "first")]

  matriz_covid_rf <- table(
    Real = factor(covid_test$MURIO, levels = c("Defunción", "Sin defunción")),
    Predicho = factor(pred_clase_covid_rf, levels = c("Defunción", "Sin defunción"))
  )
  matriz_covid_rf

  VP <- matriz_covid_rf[1,1]; FN <- matriz_covid_rf[1,2]
  FP <- matriz_covid_rf[2,1]; VN <- matriz_covid_rf[2,2]

  data.frame(
    exactitud = (VP + VN) / sum(matriz_covid_rf),
    sensibilidad = ifelse(VP + FN == 0, NA, VP / (VP + FN)),
    especificidad = ifelse(VN + FP == 0, NA, VN / (VN + FP))
  )
}


## Interpretación
Random Forest suele ser más estable que un árbol individual porque combina muchas decisiones parcialmente distintas. La comparación de métricas con el árbol del capítulo anterior permite observar esa ganancia de estabilidad.

## Materiales complementarios del capítulo

### Video del capítulo

*Video disponible en la versión web del libro.*

### Video del capítulo

Disponible en YouTube:

<https://youtu.be/KELSgYttifs>

| Recurso | Descripción | Abrir o descargar |
|---|---|---|
| Presentación en PDF | Síntesis del capítulo para lectura o exposición. | [Abrir PDF](recursos/capitulo-08/capitulo-08-random-forest-presentacion.pdf) |
| Presentación editable | Diapositivas en PowerPoint. | [Descargar PPTX](recursos/capitulo-08/capitulo-08-random-forest-presentacion.pptx) |
| Infografía | Resumen visual del capítulo. | [Abrir infografía](recursos/capitulo-08/capitulo-08-random-forest-infografia.png) |

![Infografía del capítulo 8](recursos/capitulo-08/capitulo-08-random-forest-infografia.png)

Los materiales fueron creados con apoyo de NotebookLM de Google a partir del
contenido del libro y revisados y adaptados por el autor.

## Conclusión

Random Forest mejora la estabilidad y la capacidad predictiva de un solo árbol al combinar muchos árboles. Es uno de los métodos más útiles para clasificación aplicada.

## Referencias fundamentales de Random Forest

El algoritmo Random Forest fue formalizado por @breiman2001random como un ensamble de árboles construido mediante remuestreo y selección aleatoria de variables. Una explicación didáctica y comparativa puede consultarse también en @james2021islr.
